In [6]:
# Trade Policy Uncertainty and U.S.-China FDI: Tests whether trade policy uncertainty becoming more unpredictable is associated with lower foreign investment between the U.S. and China using annual data from 2015-2024.
import pandas as pd
import statsmodels.api as sm

tpu = pd.read_csv('EPUTRADE.csv')
tpu['observation_date'] = pd.to_datetime(tpu['observation_date'])
tpu['year'] = tpu['observation_date'].dt.year

tpu_by_year = tpu.groupby('year')['EPUTRADE'].mean().reset_index()
tpu_by_year.columns = ['year', 'avg_uncertainty']
tpu_by_year

tpu_by_year['uncertainty_volatility'] = tpu_by_year['avg_uncertainty'].rolling(3).std()
tpu_by_year
years = list(range(2015, 2025))

us_to_china = pd.read_csv('us_to_china_fdi.csv', skiprows=5, nrows=2)
us_to_china.columns = ['country'] + [str(y) for y in years]
us_to_china_china_row = us_to_china[us_to_china['country'].str.strip() == 'China'].iloc[0]

china_to_us = pd.read_csv('china_to_us_fdi.csv', skiprows=5, nrows=2)
china_to_us.columns = ['country'] + [str(y) for y in years]
china_to_us_china_row = china_to_us[china_to_us['country'].str.strip() == 'China'].iloc[0]

fdi_by_year = pd.DataFrame({'year': years})
fdi_by_year['us_to_china'] = [float(us_to_china_china_row[str(y)]) for y in years]
fdi_by_year['china_to_us'] = [float(china_to_us_china_row[str(y)]) for y in years]
fdi_by_year['total_fdi'] = fdi_by_year['us_to_china'] + fdi_by_year['china_to_us']
fdi_by_year

gdp = pd.read_csv('GDP_Growth_Rate.csv')
gdp['year'] = pd.to_datetime(gdp['observation_date']).dt.year
gdp = gdp[['year', 'A191RL1A225NBEA']].rename(columns={'A191RL1A225NBEA': 'gdp_growth'})

fx = pd.read_csv('USD_CNY_Exchange_Rate.csv')
fx['year'] = pd.to_datetime(fx['observation_date']).dt.year
fx_by_year = fx.groupby('year')['DEXCHUS'].mean().reset_index()
fx_by_year.columns = ['year', 'exchange_rate']

rates = pd.read_csv('US_Federal_Funds_Rate.csv')
rates['year'] = pd.to_datetime(rates['observation_date']).dt.year
rates_by_year = rates.groupby('year')['FEDFUNDS'].mean().reset_index()
rates_by_year.columns = ['year', 'interest_rate']

controls = gdp.merge(fx_by_year, on='year').merge(rates_by_year, on='year')
controls

combined = pd.merge(tpu_by_year, fdi_by_year, on='year')
combined = pd.merge(combined, controls, on='year')
combined = combined.dropna(subset=['uncertainty_volatility'])
combined[['year', 'uncertainty_volatility', 'gdp_growth', 'exchange_rate', 'interest_rate', 'total_fdi']]

x = combined[['uncertainty_volatility', 'gdp_growth', 'exchange_rate', 'interest_rate']]
x = sm.add_constant(x)
y = combined['total_fdi']

results = sm.OLS(y, x).fit(cov_type='HAC', cov_kwds={'maxlags': 1})
print(results.summary())

                            OLS Regression Results                            
Dep. Variable:              total_fdi   R-squared:                       0.503
Model:                            OLS   Adj. R-squared:                 -0.159
Method:                 Least Squares   F-statistic:                     5.398
Date:                Sun, 16 Aug 2026   Prob (F-statistic):             0.0987
Time:                        23:05:53   Log-Likelihood:                -77.178
No. Observations:                   8   AIC:                             164.4
Df Residuals:                       3   BIC:                             164.8
Df Model:                           4                                         
Covariance Type:                  HAC                                         
                             coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------------
const                   -6.2